# 13 — External Repository Training Data

Clones external ARO repositories (plugins, applications, bundles) and extracts
high-quality training pairs from their code and tests.

For each repository:
1. **Detect type** — plugin (has `plugin.yaml`), application (has `main.aro`),
   or bundle (both).
2. **Extract code** — all `.aro`, `.yaml`, `.swift`, `.c`, `.py`, `.rs` source files.
3. **Extract tests** — all `test*.aro` files from `tests/` directories.
4. **Generate prompts in 10 styles** — each code artifact gets 10 different
   prompt phrasings covering different user personas and request formats.

**Input:**  `REPOSITORIES` array (below)
**Output:** Appended to `../data/02_knowledge/knowledge_pairs.jsonl`

## Setup

In [ ]:
import sys, importlib, json, re, textwrap, subprocess, shutil, random
from pathlib import Path
from collections import Counter, defaultdict

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

REPO_CACHE = DATA_ROOT / 'external_repos'
REPO_CACHE.mkdir(parents=True, exist_ok=True)

print(f'Config loaded | model={MODEL_ID}')
print(f'Repo cache: {REPO_CACHE}')
print(f'Pairs:      {PAIRS_FILE}')

## Repositories

Add repositories to learn from here.  Each entry is a dict with:
- `url` — git clone URL
- `branch` — branch to clone (default: `main`)
- `description` — what this repo is (used in prompt generation)

In [ ]:
REPOSITORIES = [
    {
        'url': 'https://github.com/arolang/Lowbar.git',
        'branch': 'main',
        'description': 'Underscore.js-style utility plugin with 71 qualifiers and 5 actions',
    },
    # ── Add more repositories below ──────────────────────────────────────
    # {
    #     'url': 'https://github.com/arolang/SomeApp.git',
    #     'branch': 'main',
    #     'description': 'A full-stack ARO web application',
    # },
]

print(f'Repositories to process: {len(REPOSITORIES)}')
for r in REPOSITORIES:
    print(f'  {r["url"]}')

## Load model

In [ ]:
model, tokenizer, _load_fn, mlx_generate, make_sampler = load_model()
print('Model ready.')

## Helpers

In [ ]:
# ── File extensions to collect per repo type ─────────────────────────────────
PLUGIN_CODE_EXTS  = {'.swift', '.c', '.h', '.rs', '.py', '.yaml', '.yml', '.toml'}
APP_CODE_EXTS     = {'.aro', '.yaml', '.yml', '.screen', '.json', '.toml', '.store'}
TEST_CODE_EXTS    = {'.aro'}
SKIP_FILES        = {'README.md', '.gitignore', '.gitkeep', 'LICENSE', 'Package.resolved'}
SKIP_DIRS         = {'.git', '.build', '.swiftpm', '__pycache__', 'node_modules',
                     'output', 'website', '.github', 'target'}


def chat(messages, max_tokens=600, temperature=0.5):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return mlx_generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens, verbose=False,
        sampler=make_sampler(temp=temperature),
    )


def clone_or_pull(url, branch='main'):
    """Clone a repo into REPO_CACHE, or pull if already cloned."""
    name = url.rstrip('/').split('/')[-1].replace('.git', '')
    dest = REPO_CACHE / name
    if (dest / '.git').exists():
        print(f'  Pulling {name}...')
        subprocess.run(['git', '-C', str(dest), 'pull', '--ff-only'],
                       capture_output=True, text=True, timeout=60)
    else:
        print(f'  Cloning {url}...')
        subprocess.run(['git', 'clone', '--depth', '1', '-b', branch,
                        url, str(dest)], check=True, timeout=120)
    return dest


def detect_repo_type(repo_dir):
    """Detect whether repo is a plugin, application, bundle, or unknown.
    Returns a set of types: {'plugin'}, {'application'}, {'plugin', 'application'}, etc."""
    types = set()
    # Plugin detection: has Plugins/ dir with plugin.yaml, or is itself a plugin
    if (repo_dir / 'plugin.yaml').exists():
        types.add('plugin')
    for p in repo_dir.rglob('plugin.yaml'):
        if '.git' not in p.parts:
            types.add('plugin')
            break
    # Application detection: has main.aro or any .aro files
    if (repo_dir / 'main.aro').exists():
        types.add('application')
    elif list(repo_dir.glob('*.aro')):
        types.add('application')
    # Test detection
    if list(repo_dir.rglob('test*.aro')):
        types.add('has_tests')
    return types if types else {'unknown'}


def collect_files(repo_dir, extensions, skip_dirs=SKIP_DIRS):
    """Collect files matching extensions, return {relative_path: content}."""
    files = {}
    for path in sorted(repo_dir.rglob('*')):
        if not path.is_file():
            continue
        if path.name in SKIP_FILES:
            continue
        if any(part in skip_dirs for part in path.parts):
            continue
        if path.suffix.lower() in extensions:
            rel = str(path.relative_to(repo_dir))
            try:
                files[rel] = path.read_text(errors='replace').rstrip()
            except Exception:
                pass
    return files


def collect_tests(repo_dir):
    """Collect test .aro files from tests/ directories."""
    tests = {}
    for path in sorted(repo_dir.rglob('test*.aro')):
        if any(part in SKIP_DIRS for part in path.parts):
            continue
        rel = str(path.relative_to(repo_dir))
        try:
            tests[rel] = path.read_text(errors='replace').rstrip()
        except Exception:
            pass
    return tests


def collect_plugin_source(repo_dir):
    """Collect plugin implementation source files (.swift, .c, .rs, .py)."""
    return collect_files(repo_dir, PLUGIN_CODE_EXTS)


def collect_app_source(repo_dir):
    """Collect ARO application source files (.aro, .yaml, etc.)."""
    return collect_files(repo_dir, APP_CODE_EXTS)


def format_multi_file(files, repo_name=''):
    """Format a dict of {path: content} into a multi-file code block."""
    parts = []
    ext_lang = {'aro': 'aro', 'yaml': 'yaml', 'yml': 'yaml', 'swift': 'swift',
                'c': 'c', 'h': 'c', 'rs': 'rust', 'py': 'python', 'json': 'json',
                'toml': 'toml', 'screen': 'text', 'store': 'yaml'}
    for fname in sorted(files.keys()):
        ext = Path(fname).suffix.lstrip('.')
        lang = ext_lang.get(ext, ext)
        parts.append(f'## {fname}\n```{lang}\n{files[fname]}\n```')
    return '\n\n'.join(parts)


def extract_feature_sets(code):
    """Extract individual feature set blocks from ARO code."""
    blocks = []
    pattern = r'(\([^)]+\)\s*(?:when\s+[^{]+)?\{)'
    parts = re.split(pattern, code)
    current = ''
    depth = 0
    for part in parts:
        current += part
        depth += part.count('{') - part.count('}')
        if depth == 0 and current.strip():
            cleaned = current.strip()
            if cleaned.startswith('(') and '{' in cleaned:
                blocks.append(cleaned)
            current = ''
    return blocks


def read_readme(repo_dir):
    """Read README.md if it exists."""
    readme = repo_dir / 'README.md'
    if readme.exists():
        return readme.read_text(errors='replace')
    return ''


def read_plugin_yaml(repo_dir):
    """Find and read plugin.yaml."""
    direct = repo_dir / 'plugin.yaml'
    if direct.exists():
        return direct.read_text(errors='replace')
    for p in repo_dir.rglob('plugin.yaml'):
        if '.git' not in p.parts:
            return p.read_text(errors='replace')
    return ''


print('Helpers ready.')

## Prompt styles

Each code artifact is paired with 10 different prompt styles to maximise
learning diversity.

In [ ]:
PROMPT_STYLES = [
    {
        'name': 'direct_request',
        'template': 'Write {artifact_type} that {description}.',
        'system': None,  # use default
    },
    {
        'name': 'beginner_question',
        'template': 'I\'m new to ARO. How do I {description}? Show me the code.',
        'system': None,
    },
    {
        'name': 'senior_dev',
        'template': 'Implement {artifact_type}: {description}. Follow ARO best practices.',
        'system': None,
    },
    {
        'name': 'explain_then_code',
        'template': 'Explain how to {description} in ARO, then provide the implementation.',
        'system': None,
    },
    {
        'name': 'terse',
        'template': '{description}',
        'system': None,
    },
    {
        'name': 'test_driven',
        'template': 'Write ARO tests for {description}. Use the Then action to assert expected values.',
        'system': None,
    },
    {
        'name': 'refactor',
        'template': 'Show me the best way to structure {artifact_type} that {description}.',
        'system': None,
    },
    {
        'name': 'comparison',
        'template': 'How would I {description} in ARO? Show the ARO way of doing it.',
        'system': None,
    },
    {
        'name': 'review',
        'template': 'Review this approach to {description} and show the correct ARO implementation.',
        'system': None,
    },
    {
        'name': 'documentation',
        'template': 'Document how {artifact_type} works for {description}. Include usage examples in ARO.',
        'system': None,
    },
]


def generate_styled_description(code, context, style_hint=''):
    """Ask the LLM to produce a concise natural-language description for code."""
    messages = [
        {'role': 'system', 'content': (
            'You are a technical writer. Describe what the given code does in '
            '1-2 sentences. Be specific about behaviour and data flow. '
            'Do NOT include code. Reply with ONLY the description.'
        )},
        {'role': 'user', 'content': f'{context}\n\n```\n{code[:2000]}\n```'},
    ]
    desc = chat(messages, max_tokens=120, temperature=0.3).strip()
    # Clean LLM prefixes
    desc = re.sub(r'^(This |The |It )', '', desc, count=1)
    if desc:
        desc = desc[0].lower() + desc[1:]
    return desc if len(desc) >= 15 else None


def make_prompts(description, artifact_type='an ARO feature set'):
    """Generate 10 prompt variants from PROMPT_STYLES for a given description."""
    prompts = []
    for style in PROMPT_STYLES:
        prompt = style['template'].format(
            description=description,
            artifact_type=artifact_type,
        )
        prompts.append({
            'style': style['name'],
            'instruction': prompt,
        })
    return prompts


print(f'{len(PROMPT_STYLES)} prompt styles defined.')

## Main loop — process each repository

In [ ]:
clean_notebook_pairs('NB13')

all_pairs = []
repo_stats = {}

for repo_cfg in REPOSITORIES:
    url   = repo_cfg['url']
    branch = repo_cfg.get('branch', 'main')
    repo_desc = repo_cfg.get('description', '')
    repo_name = url.rstrip('/').split('/')[-1].replace('.git', '')

    print(f'\n{"="*70}')
    print(f'  {repo_name}  ({url})')
    print(f'{"="*70}')

    # ── Clone / pull ──────────────────────────────────────────────────────
    repo_dir = clone_or_pull(url, branch)
    repo_types = detect_repo_type(repo_dir)
    print(f'  Type: {repo_types}')

    readme      = read_readme(repo_dir)
    plugin_yaml = read_plugin_yaml(repo_dir)
    pairs = []
    st = {'app_files': 0, 'plugin_files': 0, 'test_files': 0,
          'feature_sets': 0, 'test_cases': 0, 'pairs': 0}
    source_key = f'external:{repo_name}'

    # ── 1. Application code ──────────────────────────────────────────────
    if 'application' in repo_types or 'plugin' in repo_types:
        app_files = collect_app_source(repo_dir)
        st['app_files'] = len(app_files)

        if app_files:
            full_output = format_multi_file(app_files, repo_name)

            # Ground-truth pair: full app
            app_desc = repo_desc or f'a complete ARO application called {repo_name}'
            for p in make_prompts(app_desc, 'an ARO application'):
                pairs.append({
                    'instruction': p['instruction'],
                    'output': full_output,
                    'source': f'{source_key}/app/{p["style"]}',
                    'score': 1.0,
                    'metadata': {'repo': repo_name, 'type': 'app_full',
                                 'style': p['style']},
                })

            # Per-feature-set snippet pairs
            for fname, content in app_files.items():
                if not fname.endswith('.aro'):
                    continue
                blocks = extract_feature_sets(content)
                for block in blocks:
                    if len(block.strip()) < 40:
                        continue
                    desc = generate_styled_description(
                        block, f'File: {fname} in repo {repo_name}'
                    )
                    if not desc:
                        continue
                    st['feature_sets'] += 1
                    # Pick 3 random styles for snippets (avoid explosion)
                    for p in random.sample(make_prompts(desc), min(3, len(PROMPT_STYLES))):
                        pairs.append({
                            'instruction': p['instruction'],
                            'output': f'```aro\n{block}\n```',
                            'source': f'{source_key}/snippet/{fname}/{p["style"]}',
                            'score': 1.0,
                            'metadata': {'repo': repo_name, 'file': fname,
                                         'type': 'snippet', 'style': p['style']},
                        })
            print(f'  App files: {len(app_files)}, feature sets: {st["feature_sets"]}')

    # ── 2. Plugin source ─────────────────────────────────────────────────
    if 'plugin' in repo_types:
        plugin_files = collect_plugin_source(repo_dir)
        st['plugin_files'] = len(plugin_files)

        if plugin_files:
            plugin_output = format_multi_file(plugin_files, repo_name)
            plugin_desc = repo_desc or f'an ARO plugin called {repo_name}'

            for p in make_prompts(plugin_desc, 'an ARO plugin'):
                pairs.append({
                    'instruction': p['instruction'],
                    'output': plugin_output,
                    'source': f'{source_key}/plugin/{p["style"]}',
                    'score': 1.0,
                    'metadata': {'repo': repo_name, 'type': 'plugin_full',
                                 'style': p['style']},
                })

            # plugin.yaml as a standalone pair
            if plugin_yaml:
                for p in make_prompts(
                    f'configure the {repo_name} plugin manifest',
                    'a plugin.yaml'
                )[:3]:
                    pairs.append({
                        'instruction': p['instruction'],
                        'output': f'```yaml\n{plugin_yaml}\n```',
                        'source': f'{source_key}/plugin_yaml/{p["style"]}',
                        'score': 1.0,
                        'metadata': {'repo': repo_name, 'type': 'plugin_yaml',
                                     'style': p['style']},
                    })
            print(f'  Plugin files: {len(plugin_files)}')

    # ── 3. Tests ─────────────────────────────────────────────────────────
    if 'has_tests' in repo_types:
        test_files = collect_tests(repo_dir)
        st['test_files'] = len(test_files)

        # All tests as one bundle
        if test_files:
            test_output = format_multi_file(test_files, repo_name)
            for p in make_prompts(
                f'test the {repo_name} plugin/application using ARO\'s test framework',
                'ARO tests'
            ):
                pairs.append({
                    'instruction': p['instruction'],
                    'output': test_output,
                    'source': f'{source_key}/tests_full/{p["style"]}',
                    'score': 1.0,
                    'metadata': {'repo': repo_name, 'type': 'tests_full',
                                 'style': p['style']},
                })

        # Individual test files
        for tname, tcontent in test_files.items():
            blocks = extract_feature_sets(tcontent)
            st['test_cases'] += len(blocks)

            # Description per test file
            feature = Path(tname).stem.replace('test-', '').replace('test_', '').replace('-', ' ')
            desc = generate_styled_description(
                tcontent, f'Test file for {feature} in {repo_name}'
            )
            if not desc:
                desc = f'test the {feature} functionality of the {repo_name} plugin'

            # 3 styles per test file
            for p in random.sample(make_prompts(desc, 'ARO tests'), min(3, len(PROMPT_STYLES))):
                pairs.append({
                    'instruction': p['instruction'],
                    'output': f'```aro\n{tcontent}\n```',
                    'source': f'{source_key}/test/{tname}/{p["style"]}',
                    'score': 1.0,
                    'metadata': {'repo': repo_name, 'file': tname,
                                 'type': 'test_individual', 'style': p['style']},
                })

        print(f'  Test files: {len(test_files)}, test cases: {st["test_cases"]}')

    # ── 4. README knowledge (if substantial) ─────────────────────────────
    if len(readme) > 200:
        # Teach how to use this repo
        readme_desc = f'use the {repo_name} plugin/application'
        pairs.append({
            'instruction': f'How do I use {repo_name} in ARO? Explain the features and show examples.',
            'output': readme[:6000],
            'source': f'{source_key}/readme',
            'score': 0.8,
            'metadata': {'repo': repo_name, 'type': 'readme'},
        })

    # ── Save ─────────────────────────────────────────────────────────────
    save_notebook_pairs('NB13', pairs)
    all_pairs.extend(pairs)
    st['pairs'] = len(pairs)
    repo_stats[repo_name] = st
    print(f'  Total pairs: {len(pairs)}')

print(f'\nDone: {len(all_pairs)} pairs from {len(repo_stats)} repositories')

## Summary

In [ ]:
print('=' * 72)
print('External Repository Training Data — Summary')
print('=' * 72)

types = Counter(p.get('metadata', {}).get('type', 'unknown') for p in all_pairs)
styles = Counter(p.get('metadata', {}).get('style', 'unknown') for p in all_pairs)

print(f'\n{"Repo":<25}  {"App":>4}  {"Plugin":>6}  {"Tests":>5}  '
      f'{"FS":>4}  {"TC":>4}  {"Pairs":>6}')
print('\u2500' * 72)
for name, s in sorted(repo_stats.items()):
    print(f'{name:<25}  {s["app_files"]:>4}  {s["plugin_files"]:>6}  '
          f'{s["test_files"]:>5}  {s["feature_sets"]:>4}  '
          f'{s["test_cases"]:>4}  {s["pairs"]:>6}')
print('\u2500' * 72)
total = sum(s['pairs'] for s in repo_stats.values())
print(f'{"TOTAL":<25}  {"":>4}  {"":>6}  {"":>5}  {"":>4}  {"":>4}  {total:>6}')

print(f'\nBy pair type:')
for t, n in sorted(types.items(), key=lambda x: -x[1]):
    print(f'  {t:<25}: {n}')

print(f'\nBy prompt style:')
for s, n in sorted(styles.items(), key=lambda x: -x[1]):
    print(f'  {s:<25}: {n}')

print(f'\nAll pairs appended to: {PAIRS_FILE}')

In [ ]:
import csv
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_csv_out = _run_dir / '13_external_repo_training.csv'

# One row per (repo, prompt_style) combination
_style_counts: dict[tuple[str,str], int] = {}
for p in all_pairs:
    _repo = p.get('metadata', {}).get('repo', 'unknown')
    _style = p.get('metadata', {}).get('style', 'none')
    _key = (_repo, _style)
    _style_counts[_key] = _style_counts.get(_key, 0) + 1

with open(_csv_out, 'w', newline='') as _cf:
    w = csv.writer(_cf)
    w.writerow(['repo', 'pair_count', 'prompt_style'])
    for (_repo, _style) in sorted(_style_counts):
        w.writerow([_repo, _style_counts[(_repo, _style)], _style])

print(f'CSV saved: {_csv_out}  ({len(_style_counts)} rows)')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white', 'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa', 'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa', 'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
import numpy as np
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '13_external_repo_training.png'

if repo_stats:
    # ── Left: pairs by type (horizontal bar) ─────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, max(4, len(repo_stats) * 1.2 + 1)))

    _repos   = sorted(repo_stats.keys())
    _app_p   = []
    _plugin_p = []
    _test_p  = []
    _other_p = []
    for r in _repos:
        rc = Counter(p['metadata']['type'] for p in all_pairs if p['metadata']['repo'] == r)
        _app_p.append(rc.get('app_full', 0) + rc.get('snippet', 0))
        _plugin_p.append(rc.get('plugin_full', 0) + rc.get('plugin_yaml', 0))
        _test_p.append(rc.get('tests_full', 0) + rc.get('test_individual', 0))
        _other_p.append(rc.get('readme', 0))

    y = np.arange(len(_repos))
    h = 0.6
    left = np.zeros(len(_repos))
    ax1.barh(y, _app_p, h, left=left, label='Application', color='#3498db', edgecolor='white')
    left += _app_p
    ax1.barh(y, _plugin_p, h, left=left, label='Plugin', color='#9b59b6', edgecolor='white')
    left += _plugin_p
    ax1.barh(y, _test_p, h, left=left, label='Tests', color='#2ecc71', edgecolor='white')
    left += _test_p
    ax1.barh(y, _other_p, h, left=left, label='Readme', color='#e67e22', edgecolor='white')

    _totals = [a + p + t + o for a, p, t, o in zip(_app_p, _plugin_p, _test_p, _other_p)]
    for i, tot in enumerate(_totals):
        ax1.text(tot + 0.5, i, str(tot), va='center', ha='left', fontsize=9, color='#333')

    ax1.set_yticks(y)
    ax1.set_yticklabels(_repos, fontsize=9)
    ax1.set_xlabel('Training pairs')
    ax1.set_title('Pairs by repository', fontsize=11, fontweight='bold')
    ax1.legend(fontsize=8, loc='lower right')
    ax1.grid(axis='x', alpha=0.3)

    # ── Right: prompt style distribution (pie) ────────────────────────────
    if styles:
        _slabels = sorted(styles.keys(), key=lambda s: -styles[s])
        _svals   = [styles[s] for s in _slabels]
        _scolors = plt.cm.Set3(np.linspace(0, 1, len(_slabels)))
        wedges, texts = ax2.pie(
            _svals, labels=_slabels, colors=_scolors,
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
        )
        for t in texts:
            t.set_fontsize(7)
        ax2.add_artist(plt.Circle((0, 0), 0.45, fc='white'))
        ax2.text(0, 0, f'{sum(_svals)}\npairs', ha='center', va='center',
                 fontsize=12, fontweight='bold', color='#2c3e50')
    ax2.set_title('Prompt style distribution', fontsize=11, fontweight='bold')

    fig.suptitle(
        f'External Repo Training \u2014 {len(all_pairs):,} pairs from {len(repo_stats)} repos',
        fontsize=13, fontweight='bold',
    )
    fig.tight_layout()
    fig.savefig(_out)
    plt.close(fig)
    print(f'Saved: {_out}')
else:
    print('No data to plot.')